In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

transactions = pd.read_csv("../data/transaction_data.csv")

# Assignment 2 - Safety Stock Calculation

## Introduction

In the first assignment, we inspected and cleaned the transaction dataset so it could be used for a safety stock analysis. The dataset had 760,460 transactions and 9 columns. The first step was renaming some of the columns (sku_numb3r, Manufacturing Site and division Code) so that they matched the metadata and followed the same naming convention.

When inspecting the numeric columns, we found a few problems. Some lead times had two extra zeroes added (2100 and 2800 instead of 21 and 28), so we corrected them. A few unit prices were negative but otherwise looked like normal prices, so we changed them to positive values. The order_quantity column also had negative values. In Assignment 1 we kept them, but based on the feedback we received, these are most likely data entry errors with an incorrect negative sign so in this assignment we convert them to positive values with `.abs()`.

For missing values, we dropped the rows with no SKU number, inventory type or stocking type, and filled missing manufacturing sites and division codes with "Unknown". The feedback on Assignment 1 pointed out that filling with "Unknown" is the safer option for all columns, because the data has several layers of information (SKU, type, site, division) and dropping a row because one of them is missing removes the information in the others too. In this assignment we will fill all missing values with "Unknown". 

Lastly, we filtered the data to keep only finished goods (FG) that are make to stock (MTS), because these are the items safety stock can be calculated for. In this filtered data we found **360 unique SKUs**, 20 manufacturing sites and 55 divisions (both counts include "Unknown").

A few of these findings matter for this assignment:
- The analysis is again done on the FG and MTS subset, so we should end up with safety stock values for 360 SKUs.
- The lead times are now only 14, 21 or 28 days, which matters because lead time is part of the safety stock formula.
- The negative order quantities are now converted to positive values. This is important because the standard deviation of order quantity is part of the safety stock formula.

In this assignment we will calculate how much safety stock should be kept for each SKU, so that the company can handle changes in demand. To do this, we will:
1. Filter the cleaned data for the right SKUs and aggregate it by SKU, calculate the minimum, maximum, average, median, variance and standard deviation of order quantity, along with the average lead time.
2. Use this data to calculate safety stock for each SKU at service levels of 75%, 90% and 95%, using the provided formula.
3. Look at the safety stock values at the 95% service level to find the SKUs with the largest and smallest safety stock, and the average safety stock across all SKUs.

## Data cleaning

Since this is a new notebook, we first need to repeat the cleaning steps from Assignment 1 so that we work with the same cleaned data. We rename the columns to match the metadata, correct the lead times that had extra zeroes and change the negative unit prices and order quantities to positive values. For missing values, we change our approach from Assignment 1. Instead of dropping rows with a missing SKU number, inventory type or stocking type, we fill all missing values with "Unknown", as suggested in the feedback. This way no rows are lost. Rows with "Unknown" values that aren't suitable for the safety stock calculation are removed later, when we filter the data in Question 1.   

The lead time fix is important here, because lead time is used directly in the safety stock formula, and values like 2800 would make the results too high. Unit price isn't used in this assignment, but we still fix it so the dataset is consistent with the first assignment. Some steps from Assignment 1 are not repeated, because they aren't needed for the safety stock calculation. We don't extract the country codes or calculate the total sales value, since this assignment only needs the SKU number, order quantity and lead time.  

The cleaned dataset could have been exported from the previous assignment, but the instructions were unclear if we can do that, so we just repeat the same steps. 

In [2]:
# Renaming columns
transactions.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price",
                        "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

# Fixing outliers from the columns
transactions["lead_time"] = transactions["lead_time"].replace({2100:21, 2800:28})
transactions["unit_price"] = transactions["unit_price"].abs()

#Making negative values in order_quantity column to positive based on assignment 1 feedback
transactions["order_quantity"] = transactions["order_quantity"].abs()

# Filling all missing values with "Unknown" based on assignment 1 feedback
transactions_clean = transactions.fillna("Unknown")

transactions_clean.sample(5)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
745328,0EC761AF,RM,MTS,28,989.653002,JP-5C,AB2F,2025-12-09,113
742128,8794B739,WIP,MTS,28,1002.532591,Unknown,Unknown,2025-12-04,74
302302,C7864E32,WIP,MTO,28,986.282100,CA-5A,CCE6,2024-03-12,94
709343,F2BC5D9D,RM,MTO,28,985.541863,CA-8A,8EBD,2025-10-18,73
272213,B966BB2A,FG,ATO,21,192.628783,Unknown,DFD7,2024-01-28,6


## Question 1 — Required Transformation
*Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.*

The safety stock formula needs the standard deviation of demand and the lead time for each SKU, but at the moment our data has one row for every transaction. That means we first need to group the transactions by SKU and summarise them, so that each SKU has its own row with the statistics we need.

We chose `.groupby()` together with `.aggregate()` to do this, which also enables us to calculate all the required statistics in one step, instead of making a separate table for each one and merging them together. We only use the FG and MTS data, because make-to-stock finished goods are the only items we can calculate safety stock for. We also add a third condition to filter out the "Unknown" SKU because otherwise `.groupby("sku_number")` would treat all the unknown SKUs as one product and calculate a safety stock for it, even though these rows most likely belong to many different products.

In [3]:
#We filter to see only FG and MTS stock that also do not have an "Unknown" SKU
filtered_data = transactions_clean[(transactions_clean["inventory_type"] == "FG") & (transactions_clean["stocking_type"] == "MTS") & 
                                   (transactions_clean["sku_number"] != "Unknown")]

sku_stats = (
    filtered_data
    .groupby("sku_number")
    .aggregate({"order_quantity": ["min", "max", "mean", "median", "var", "std"], "lead_time": "mean"})
)
sku_stats.columns = ["min_quantity", "max_quantity", "avg_quantity", "median_quantity", "var_quantity", "std_quantity", "avg_lead_time"]
sku_stats

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time
sku_number,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000
...,...,...,...,...,...,...,...
F9ED9C68,0,38,12.147239,11.0,68.706582,8.288943,28.000000
FACEC76F,0,29,8.147727,7.0,36.104363,6.008691,14.000000
FAF5C442,0,87,20.793792,17.0,231.350717,15.210218,19.711752


- For **order_quantity**, we give a list of calculations instead of just one: "min" (minimum), "max" (maximum), "mean" (average), "median", "var" (variance) and "std" (standard deviation). This way all six statistics are calculated in one step instead of writing a separate line for each.
- For **lead_time**, we only need the "mean". Some SKUs have more than one lead time value in the data, so the average gives us one lead time per SKU that we can use in the safety stock formula.

We use **sku_stats.columns = [...]** to rename the calculation columns. The new names have to be in the same order as the calculations in the aggregate function.

## Question 2- Safety Stock Calculation
*Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the ppf function of SciPy.Stat module of the SciPy package, or simply use the numeric values from any standard normal distribution function.*

In this question, we use the transformed data from Question 1 to calculate three safety stock values for each SKU, for service level coefficients of 75%, 90% and 95%. As explained in the assignment instructions, safety stock is calculated with the following formula:

**Safety stock = z × o × √L**

- **z** is the z-score of the service level coefficient (how many standard deviations of extra stock we need to keep above average demand, so that we can fill orders as often as the service level promises).
- **o** is the standard deviation of demand, which is the std_quantity column we calculated in Question 1,
- **L** is the lead time in days, which is the avg_lead_time column from Question 1.

The standard deviation and lead time are already in our data, but we still need the z-scores. To find them we use the **SciPy** package, as suggested in the assignment instructions (SciPy Developers, n.d. -a).

In [4]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)
z_75, z_90, z_95

(np.float64(0.6744897501960817),
 np.float64(1.2815515655446004),
 np.float64(1.6448536269514722))

To find the z-scores, we use the `norm` object from the `scipy.stats`, which we imported in the set-up cell. **norm** is for the normal distribution. If we don't give it a mean or standard deviation, it uses the **standard** normal distribution, which has a mean of 0 and a standard deviation of 1. 

The function we use is `.ppf()`, (percent point function) (Tutorials Point, n.d.). If we want to have enough stock 95% of the time we need to answer what demand level is higher than 95% of days.

`norm.ppf(0.95)` tells us how high we need to go so that 95% of the normal distribution will be below the cutoff. The answer is the z-score for the 95% service level.

The result is about **1.645**, which means we need to keep enough stock to cover demand up to 1.645 standard deviations above the average. This way we will have enough safety stock for 95% of days and we would only run out on 5% of days that have really high demand. 

The same logic applies to the other two service levels. `norm.ppf(0.75)` gives about **0.674** and `norm.ppf(0.90)` gives about **1.282**. The higher the service level, the higher the z-score. If we want to be more sure that we won't run out of stock, we need a bigger stock above average demand.

We save the results in **z_75**, **z_90** and **z_95** so we can use them in the formula. 

The z-score goes up as the service level goes up: the more certain we want to be that we won't run out of stock, the more standard deviations of extra stock we need. Now that we have the z scores, and we have other parts of the formula already calculated in the `sku_stats` table, we can use it to calculate the safety stock. 

In [5]:
sku_stats["safety_stock_75"] = z_75 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["safety_stock_90"] = z_90 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["safety_stock_95"] = z_95 * sku_stats["std_quantity"] * np.sqrt(sku_stats["avg_lead_time"])
sku_stats.head()

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000,70.786909,134.497334,172.625460
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000,48.870353,92.855196,119.178354
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319,111.722959,212.277106,272.454716
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000,17.251996,32.779330,42.071814
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000,71.862125,136.540280,175.247552


In this code, we calculate the safety stock for each SKU using the formula, and we save the results in three new columns, one for each service level.

- **z_75**, **z_90** and **z_95** are the z-scores we found. Each one is a single number, so it is the same for every SKU.
- `sku_stats["std_quantity"]` selects the whole standard deviation column.
- `np.sqrt(sku_stats["avg_lead_time"])` takes the square root of the average lead time column. We use the NumPy function `np.sqrt()` to calculate the square root of the average lead time. 

Pandas does the calculation row by row, so each SKU uses its own standard deviation and lead time and gets its own safety stock value. 

The table shows that the safety stock grows as the service level goes up. For example, SKU 00D0DF3B needs about 70 units for 75%, 134 units for 90% and 172 units for 95%. Going from 75% to 95% more than doubles the safety stock, so reaching a higher service level is more expensive when it comes to inventory.  

Variability could also affect safety stock because uncertain demand requires more safety stock to be kept to avoid stockouts when demand is high. 

## Question 3 — Safety Stock Distribution

*Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.*

*Which SKU has the largest safety stock?*  
*Which SKU has the smallest safety stock?*  
*What is the average safety stock value across all SKUs?*  

### Largest safety stock  

To see the item with the largest safety stock for the 95% service level coefficient, we need to sort that column in the table we created from largest to smallest, and only display the top result. 

In [6]:
sku_stats.sort_values("safety_stock_95", ascending=False).head(1)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.25607,270.29097,346.914705


The SKU with the largest safety stock at the 95% service level is **A4287F10**, with a safety stock of about **347 units**.

It has the largest safety stock because it has high demand variability and a long lead time. Its standard deviation is 41.19, the highest among all SKUs, and its order quantities range a lot. Its average lead time is about 26 days, close to the maximum of 28, so it takes a long time to restock and the safety stock has to cover a long period of time. This SKU also has high demand in general, with an average order quantity of about 81 units and a median of 93. Since demand is both high and unpredictable, the company would need to keep a large amount of extra stock for this product to meet orders 95% of the time.  

We compare with other top 5 to look at all the measures and determine why this has such high safety stock.

In [7]:
sku_stats.sort_values("std_quantity", ascending=False).head(5)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.256070,270.290970,346.914705
58023D4F,0,165,79.365556,91.0,1623.838683,40.296882,26.430544,139.733279,265.497588,340.762465
EAEE18E6,0,164,83.595689,96.0,1609.696055,40.121018,26.447866,139.169034,264.425507,339.386464
A093EB12,0,166,81.193206,94.0,1539.376844,39.234893,25.874735,134.612621,255.768179,328.274904
361B3A3C,0,189,85.367415,96.0,1531.183775,39.130343,25.645485,133.657850,253.954085,325.946539


All the top SKUs share varying order quantity and a long average lead time. This means that safety stock has to cover longer periods of time for an uncertain demand, which is why the safety stock is so high. 

We can also look at how these SKUs compare with others. 

In [8]:
sku_stats[["std_quantity", "avg_lead_time", "avg_quantity"]].describe()

,std_quantity,avg_lead_time,avg_quantity
count,360.000000,360.000000,360.000000
mean,13.107914,19.667821,35.658702
std,8.526981,6.195549,38.708815
min,5.690555,14.000000,7.560000
25%,6.669749,14.000000,8.971138
50%,7.620715,14.000000,10.332843
75%,19.706724,27.386968,91.148475
max,41.192615,28.000000,101.650199


A4287F10 has the highest standard deviation of all 360 SKUs (41.19), which is more than three times the average of 13.1. Its average lead time (26.2 days) and average order quantity (81.0 units) are also well above the average across all SKUs (19.7 days and 35.7 units).

### Smallest safety stock  

We use the same code as the previous question, but remove `ascending = False` so that the values are sorted from smallest to largest. We only view the top result to answer the question: Which SKU has the smallest safety stock?

In [9]:
sku_stats.sort_values("safety_stock_95").head(1)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95
sku_number,,,,,,,,,,
83AE2839,0,33,7.868132,7.0,32.382418,5.690555,14.0,14.361308,27.286934,35.0224


The SKU with the smallest safety stock at the 95% service level is **83AE2839**, with a safety stock of about **35 units**.

It has the smallest safety stock for the opposite reasons to A4287F10. Its standard deviation (5.69) is the lowest of all 360 SKUs, as the minimum in the describe table shows, so its demand is the most stable. Its lead time is 14 days, the shortest in the data, so it can be restocked quickly and the safety stock only needs to cover a short period.

This SKU also has low demand, with an average order quantity of about 8 units and a median of 7. Its safety stock of 35 units is about four and a half times its average order quantity, because even small orders still vary from 0 to 33 units.

### Average safety stock across all SKUs  

To calculate this we use the `.mean()` function on the `safety_stock_95` column. 

In [10]:
sku_stats["safety_stock_95"].mean()

np.float64(102.02559297184781)

In [11]:
sku_stats["safety_stock_95"].describe()

count    360.000000
mean     102.025593
std       78.827074
min       35.022400
25%       41.048831
50%       49.145774
75%      171.522346
max      346.914705
Name: safety_stock_95, dtype: float64

The average safety stock across all 360 SKUs at the 95% service level is about **102 units**.

However, using `.describe()` on the `safety_stock_95` column, we can see that the median is only about 49 units, and 75% of SKUs need around 172 units or less. The mean is much higher than the median because a smaller group of SKUs with high demand variability and long lead times pull the average up.

This shows that the SKUs fall into very different groups. In practice, it is best to look at safety stock for each SKU separately instead of using one average value for all products.

## Side Quest - Non-parametric safety stock

The safety stock formula we used in Question 2 is one of the most common methods in practice, but it assumes that the demand for each SKU follows a normal distribution. Real demand is often highly variable. In this section, we calculate the safety stock again with a non-parametric method, which means it doesn't assume any specific distribution, and compare the results with the normal formula.

Normal stock covers the average demand, and safety stock is the extra stock on top of that for days when demand is higher than average. So safety stock tells us how far above the average demand do we need to be ready for days with higher demand. 

For a 95% service level, we want a cut-off that demand stays below on 95% of days, and the safety stock is the gap between the average demand and this cut-off. If demand is normally distributed, the 95% cut-off is always 1.645 standard deviations above the average.

Before making the calculation, we researched quantiles and empirical quantiles. A quantile (or percentile) is a cut-off point in ordered data, where a certain share of the values falls below it. For example, 70% of the data falls below the 70th percentile (NIST/SEMATECH, n.d.). An **empirical quantile** is a quantile that we find by looking at the actual data, instead of calculating it from a formula that assumes a certain distribution. 

Instead of guessing the cut-off from the mean and standard deviation, we find it directly in the real orders of each SKU, and then calculate the same gap between the cut-off and the average demand. This gap replaces **z × o**. Both measure how far above the average the cut-off is, but one assumes a normal distribution and the other uses the real data. The lead time still works the same way, because the safety stock has to cover the whole time it takes to restock, so we keep multiplying by the square root of the lead time, like in the formula from the assignment instructions. 

**Non-parametric safety stock = (empirical quantile − average demand) × √L**

To find the empirical quantiles, we use the **mquantiles** function from the **scipy.stats.mstats** module, as suggested in the assignment instructions (SciPy Developers, n.d.-b).

### Calculating empirical quantiles  

In order to calculate the empirical quantiles for each SKU, we need to create our own functions that look at each SKU and give back its quantile. We need these functions because `.aggregate()` doesn't have a built-in option for quantiles like it does for "mean" or "std". We need to import the package that allows us to use this formula, so we add it to the import cell at the beginning of the notebook. `from scipy.stats.mstats import mquantiles` 

In [12]:
# Functions that give the 75%, 90% and 95% empirical quantile of a column
def quantile_75(x):
    return mquantiles(x, prob=[0.75])[0]

def quantile_90(x):
    return mquantiles(x, prob=[0.90])[0]

def quantile_95(x):
    return mquantiles(x, prob=[0.95])[0]

Using `def` we start creating a new function. We choose the names `quantile_75`, `quantile_90` and `quantile_95` to represent the results.  

We use `x` as a placeholder for any data that we use the function on (which will be all the order quantities of one SKU).  

As per the assignment instructions we use `mquantiles(x, prob=[0.75/0.90/0.95])` to calculate the corresponding empirical quantiles of `x`. `[0]` takes the first answer in the list because the function returns its answer as a list. 

Since we want to calculate the quantiles of order_quantity for each SKU, we use the `groupby` and `aggregate` functions. We group the filtered data by `sku_number` and then apply our quantile functions to the `order_quantity` column.

In [13]:
# Calculating the quantiles of order quantity for each SKU
sku_quantiles = (
    filtered_data
    .groupby("sku_number")
    .aggregate({"order_quantity": [quantile_75, quantile_90, quantile_95]})
)
sku_quantiles.columns = ["q75_quantity", "q90_quantity", "q95_quantity"]
sku_quantiles.head()

,q75_quantity,q90_quantity,q95_quantity
sku_number,,,
00D0DF3B,113.0,124.0,132.00
01083EC9,33.3,46.0,52.54
036A78DC,111.0,125.0,133.00
0469CB75,13.0,19.0,22.00
047A0938,114.0,125.0,134.00


The quantiles are not the safety stock themselves, but cut-offs in the demand. For example, q95_quantity tells us the order size that 95% of the orders don't go above. To get the safety stock, we still need to subtract the average demand (because normal stock already covers that) and multiply by the square root of the lead time, which we do in the next step. We merge the sku_quantiles to our sku_stats tables for further calculations. 

In [14]:
sku_stats = pd.merge(sku_stats, sku_quantiles, on = "sku_number")
sku_stats.head()

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity
sku_number,,,,,,,,,,,,,
00D0DF3B,35,161,99.988852,101.0,393.366301,19.833464,28.000000,70.786909,134.497334,172.625460,113.0,124.0,132.00
01083EC9,0,82,23.020997,20.0,249.989032,15.811041,21.000000,48.870353,92.855196,119.178354,33.3,46.0,52.54
036A78DC,0,163,92.307971,98.0,1021.387756,31.959158,26.862319,111.722959,212.277106,272.454716,111.0,125.0,133.00
0469CB75,0,36,8.934959,8.0,46.730446,6.835967,14.000000,17.251996,32.779330,42.071814,13.0,19.0,22.00
047A0938,20,159,100.031720,100.0,405.407122,20.134724,28.000000,71.862125,136.540280,175.247552,114.0,125.0,134.00


### Non-parametric safety stock 

We add three new columns to the sku_stats table which calculate the non parametric safety stock by subtracting the average demand from the calculated quantiles. This is how far above average demand the empirical quantile is, which is the extra stock we need to cover demand at that service level. We still multiply by the square root of the lead time to adjust for the number of days needed to restock. We then show the normal and non-parametric safety stock side by side to compare them.

In [15]:
sku_stats["np_safety_stock_75"] = (sku_stats["q75_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["np_safety_stock_90"] = (sku_stats["q90_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats["np_safety_stock_95"] = (sku_stats["q95_quantity"] - sku_stats["avg_quantity"]) * np.sqrt(sku_stats["avg_lead_time"])
sku_stats[["safety_stock_95", "np_safety_stock_95"]].head()

,safety_stock_95,np_safety_stock_95
sku_number,,
00D0DF3B,172.625460,169.387075
01083EC9,119.178354,135.273064
036A78DC,272.454716,210.902193
0469CB75,42.071814,48.884906
047A0938,175.247552,179.743245


The table shows that the two methods give similar safety stock for some SKUs but quite different results for others. For 00D0DF3B and 047A0938, the difference is small (172.6 vs 169.4 and 175.2 vs 179.7 units), so the normal assumption works almost as well for these items.

For 036A78DC, the normal formula gives a much higher safety stock than the non-parametric method (272.5 vs 210.9 units), so the company would keep about 62 units more than the real demand. For the smaller SKUs 01083EC9 and 0469CB75, it is the other way around: the normal formula gives less safety stock, so relying on this could lead to stockouts more often.

Normal assumption can either overestimate or underestimate the safety stock, depending on the SKU. The non-parametric safety stock doesn't assume that demand follows a normal distribution. It uses the actual order quantities of each SKU, so the real shape of each SKU's demand is reflected in the result.

### Comparison of the methods  

In [16]:
sku_stats[["safety_stock_95", "np_safety_stock_95"]].describe()

,safety_stock_95,np_safety_stock_95
count,360.000000,360.000000
mean,102.025593,101.586381
std,78.827074,62.593608
min,35.022400,34.276871
25%,41.048831,47.837535
50%,49.145774,62.609748
75%,171.522346,169.550991
max,346.914705,265.456005


In [17]:
sku_stats.sort_values("np_safety_stock_95", ascending = False).head(3)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity,np_safety_stock_75,np_safety_stock_90,np_safety_stock_95
sku_number,,,,,,,,,,,,,,,,
58023D4F,0,165,79.365556,91.0,1623.838683,40.296882,26.430544,139.733279,265.497588,340.762465,108.0,122.0,131.0,147.211521,219.186425,265.456005
A4287F10,0,171,81.038513,93.0,1696.831554,41.192615,26.215139,142.256070,270.290970,346.914705,110.0,123.0,132.0,148.284907,214.845845,260.926495
EAEE18E6,0,164,83.595689,96.0,1609.696055,40.121018,26.447866,139.169034,264.425507,339.386464,111.0,124.0,131.0,140.933493,207.789228,243.788471


In [18]:
sku_stats.sort_values("np_safety_stock_95").head(3)

,min_quantity,max_quantity,avg_quantity,median_quantity,var_quantity,std_quantity,avg_lead_time,safety_stock_75,safety_stock_90,safety_stock_95,q75_quantity,q90_quantity,q95_quantity,np_safety_stock_75,np_safety_stock_90,np_safety_stock_95
sku_number,,,,,,,,,,,,,,,,
BCBD9108,0,31,8.879121,9.0,37.440781,6.118887,14.0,15.442294,29.340841,37.658562,12.0,16.48,18.04,11.677260,28.439886,34.276871
F824ED29,0,23,10.142857,10.0,38.229323,6.182987,14.0,15.604062,29.648205,38.053059,15.0,18.88,20.00,18.173764,32.691395,36.882051
56DB54B0,0,35,8.925532,7.5,47.553535,6.895907,14.0,17.403267,33.066749,42.440713,14.0,17.00,18.89,18.986921,30.211893,37.283626


On average, the two methods give almost the same safety stock (about 102 units), but the rest of the describe table shows clear differences. The non-parametric safety stock is less spread out: its maximum is much lower (about 265 compared to 347 units), while its median and 25% values are higher. So the normal formula gives more safety stock to the SKUs at the top, and less to the smaller SKUs in the middle of the distribution.

The SKUs with the largest non-parametric safety stock are the same high-demand, high-variability SKUs with long lead times that had the largest normal safety stock, including A4287F10. So both methods agree on which products need the most safety stock, but the normal formula overestimates it. 

At the bottom, the smallest safety stock values are very similar for both methods, and the SKUs with the smallest non-parametric safety stock are again low-demand SKUs with stable orders and a 14-day lead time. However, they are not the same SKUs that had the smallest normal safety stock, which shows that the two methods can rank individual SKUs differently.

Overall, the normal formula seems to overestimate safety stock for the high-demand SKUs and underestimate it for many of the smaller ones, while the non-parametric method follows the real demand of each SKU.

## References  

- NIST/SEMATECH. (n.d.). *7.2.6.2. Percentiles*. In *NIST/SEMATECH e-Handbook of Statistical Methods*. National Institute of Standards and Technology. Retrieved November 18, 2026, from https://itl.nist.gov/div898/handbook/prc/section2/prc262.htm

- SciPy Developers. (n.d.-a). *scipy.stats.norm*. SciPy documentation. Retrieved November 14, 2026, from https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html

- SciPy Developers. (n.d.-b). *scipy.stats.mstats.mquantiles*. SciPy documentation. Retrieved November 18, 2026, from https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.mquantiles.html

- Tutorials Point. (n.d.). *SciPy - stats.norm.ppf() function*. Retrieved November 14, 2026, from https://www.tutorialspoint.com/scipy/scipy_stats_norm_ppf_function.htm